# 03 — BETO fine-tuning sobre corpus MULTI-fuente (Coello-Guilarte + 3 corpus traducidos)
**Autores:** Giuliano Crenna, Bruno Emmanuel Pace (UGR)
**Etapa:** 4 — Modelado (Etapa 2b — Multi-corpus)
**Hardware:** GPU local RTX 3060 Ti 8GB VRAM
**Entorno:** system Python (CUDA), kernel `python3 (system-CUDA)`

Re-entrena BETO (`dccuchile/bert-base-spanish-wwm-cased`) sobre el **corpus unificado multi-fuente** generado por `merge_corpora.py`. La pipeline previa (`02_finetune_beto_local.ipynb`) entrenó solo sobre Coello-Guilarte 2019; este notebook entrena sobre Coello-Guilarte + Kaggle SDD + Reddit MH Posts + Depression Reddit (todos en español vía MarianMT).

**Cambios vs `02_finetune_beto_local.ipynb`**:
- `OUTPUT_DIR = "models/beto_multi"` (separado del mono).
- `model = "beto_multi"`, `feature_set = "beto_transformer_multi"` en el CSV unificado.
- Evalúa sobre el split multi (mismas proporciones 70/10/20 user-level).
- Idéntica config de hiperparámetros (3 epochs, lr=2e-5, batch=16, max_length=128, fp16).

**Pre-requisito**:
- Haber ejecutado la pipeline multi-corpus:
  1. `python -m src.data.project_user_csvs`
  2. `python -m src.translation.translate run --source <src> --backend opus_mt --limit 80000` (×3)
  3. `python -m src.data.merge_corpora`
  4. `python -m src.data.build_splits --out ./data/processed/splits_multi`

In [ ]:
# Parámetros (papermill overrides con -p SEED ... -p EPOCHS ... -p BATCH_SIZE ...)
SEED = 42
DOWNSAMPLE_SEED = 42
MAX_LENGTH = 128
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
EPOCHS = 3
OUTPUT_DIR = "models/beto_multi"
DATA_PROC = "data/processed/splits_multi"

In [ ]:
import os, sys, time
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "src").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

assert torch.cuda.is_available(), (
    "CUDA no disponible. Este notebook requiere GPU. "
    f"torch={torch.__version__}, cuda={torch.cuda.is_available()}"
)
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"torch: {torch.__version__} | cuda: {torch.__version__.split('+')[-1]}")

from src.utils.seeds import set_seed
from src.models.train_baseline import load_split_data
from src.models.train_beto import run_training
from src.models.predict import predict_texts
from src.evaluation.metrics import compute
from sklearn.metrics import confusion_matrix, roc_auc_score

set_seed(SEED)
OUTPUT_DIR = REPO_ROOT / OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR = REPO_ROOT / "reports" / "figures"
TAB_DIR = REPO_ROOT / "reports" / "tables"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROC = REPO_ROOT / DATA_PROC
print(f"Repo: {REPO_ROOT}")
print(f"Output: {OUTPUT_DIR}")
print(f"Splits: {DATA_PROC}")

## 1. Carga de splits user-level (corpus multi-fuente)

**Pre-requisito**: haber corrido `python -m src.data.build_splits --out ./data/processed/splits_multi`. Si falta, abortar.

In [ ]:
assert (DATA_PROC / "train.parquet").exists(), (
    f"No se encontró {DATA_PROC / 'train.parquet'}. "
    "Ejecutá antes: python -m src.data.build_splits --out data/processed/splits_multi"
)
df_train, df_val, df_test = load_split_data(data_dir=DATA_PROC, max_samples=None, seed=SEED)

print(f"Train: {len(df_train):,} | {df_train['label'].value_counts().to_dict()}")
print(f"Val:   {len(df_val):,} | {df_val['label'].value_counts().to_dict()}")
print(f"Test:  {len(df_test):,} | {df_test['label'].value_counts().to_dict()}")
print(f"\nPor fuente (train): {df_train['corpus_source'].value_counts().to_dict()}")
print(f"Por fuente (val):   {df_val['corpus_source'].value_counts().to_dict()}")
print(f"Por fuente (test):  {df_test['corpus_source'].value_counts().to_dict()}")
print("OK — splits cargados")

## 2. Downsampling del train (50/50)

BETO se entrena sobre train balanceado para evitar weighted loss / sampler. Val y test se mantienen intactos (distribución real).

In [ ]:
df_pos = df_train[df_train["label"] == 2]
df_neg = df_train[df_train["label"] == 0].sample(n=len(df_pos), random_state=DOWNSAMPLE_SEED)
df_down = (
    pd.concat([df_pos, df_neg])
    .sample(frac=1.0, random_state=DOWNSAMPLE_SEED)
    .reset_index(drop=True)
)
print(f"Downsampled train: {len(df_down):,} | {df_down['label'].value_counts().to_dict()}")
print(f"Por fuente (down): {df_down['corpus_source'].value_counts().to_dict()}")

## 3. Fine-tuning de BETO

Estimación de tiempo en RTX 3060 Ti 8GB (batch=16, max_length=128, fp16): depende del tamaño del nuevo train downsampled. Como referencia, 424k → ~50 min para 3 epochs.

In [ ]:
cfg_path = REPO_ROOT / "configs" / "models.yaml"
import yaml
cfg = yaml.safe_load(open(cfg_path, encoding="utf-8")) if cfg_path.exists() else {}
beto_cfg = cfg.get("beto", {})
MODEL_NAME = beto_cfg.get("model_name", "dccuchile/bert-base-spanish-wwm-cased")
print(f"Modelo: {MODEL_NAME}")
print(f"Hiperparámetros: lr={LEARNING_RATE}, batch={BATCH_SIZE}, epochs={EPOCHS}, max_length={MAX_LENGTH}")

t0 = time.time()
trainer, test_metrics = run_training(
    df_train=df_down,
    df_val=df_val,
    df_test=df_test,
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    eval_batch_size=EVAL_BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    epochs=EPOCHS,
    output_dir=str(OUTPUT_DIR),
    seed=SEED,
)
train_min = (time.time() - t0) / 60
print(f"\nEntrenamiento completo en {train_min:.1f} min")
print(f"Test metrics (Trainer.evaluate): {test_metrics}")

import json
log_history = trainer.state.log_history
with open(OUTPUT_DIR / "trainer_log_history.json", "w", encoding="utf-8") as f:
    json.dump(log_history, f, indent=2, default=str)
print(f"Log history guardado en {OUTPUT_DIR / 'trainer_log_history.json'}")

## 4. Evaluación post-hoc con el schema unificado

In [ ]:
best_model_dir = OUTPUT_DIR / "best_model"

# Eval batched via Trainer.predict (evita OOM de pasar TODO el dataset de una).
from src.models.evaluate_beto_multi import _TextDS
import json as _json
from transformers import AutoModelForSequenceClassification as _AutoM, AutoTokenizer as _AutoT, Trainer as _Trainer, TrainingArguments as _TA

_meta = _json.loads((best_model_dir.parent / "model_meta.json").read_text())
_label2id = {int(k): int(v) for k, v in _meta["label2id"].items()}
_id2label = {int(k): int(v) for k, v in _meta["id2label"].items()}
_max_len = int(_meta.get("max_length", MAX_LENGTH))
_tok = _AutoT.from_pretrained(str(best_model_dir))
_mdl = _AutoM.from_pretrained(str(best_model_dir)).to("cuda").eval()
_eval_args = _TA(output_dir=str(OUTPUT_DIR / "_tmp_eval"), per_device_eval_batch_size=EVAL_BATCH_SIZE, report_to="none", seed=SEED)
_trainer = _Trainer(model=_mdl, args=_eval_args, tokenizer=_tok)

rows = []
for split_name, df_split in [("val", df_val), ("test", df_test)]:
    t0 = time.time()
    texts = df_split["text_clean"].astype(str).tolist()
    labels_orig = df_split["label"].astype(int).tolist()
    labels_mapped = [_label2id[l] for l in labels_orig]
    ds = _TextDS(texts, labels_mapped, _tok, _max_len)
    out = _trainer.predict(ds)
    elapsed = time.time() - t0
    logits = out.predictions
    y_true = labels_orig
    y_pred_idx = np.argmax(logits, axis=1)
    y_pred = [_id2label[int(p)] for p in y_pred_idx]
    exp = np.exp(logits - logits.max(axis=1, keepdims=True))
    proba = exp / exp.sum(axis=1, keepdims=True)
    pos_idx = 1 if _id2label.get(1) == 2 else 0
    auc_bin = float(roc_auc_score(y_true, proba[:, pos_idx]))
    m = compute(y_true, y_pred, y_proba=proba.tolist())
    print(f"  {split_name}: predict batched sobre {len(texts):,} ejemplos en {elapsed:.1f}s")
    rows.append({
        "model": "beto_multi",
        "split": split_name,
        "n_train_rows": len(df_down),
        "class_balance_strategy": "downsample_majority",
        "feature_set": "beto_transformer_multi",
        "accuracy": m["accuracy"],
        "f1_macro": m["f1_macro"],
        "f1_weighted": m["f1_weighted"],
        "kappa": m["kappa"],
        "auc_binary_depressive": auc_bin,
        "f1_class_0_control": m["report"].get("0", {}).get("f1-score", np.nan),
        "f1_class_2_depressive": m["report"].get("2", {}).get("f1-score", np.nan),
        "created_at": pd.Timestamp.utcnow().isoformat(timespec="seconds"),
    })

df_metrics = pd.DataFrame(rows)
print(df_metrics.to_string(index=False))

## 5. Persistencia de artefactos

In [ ]:
unified_csv = TAB_DIR / "all_models_metrics.csv"
df_existing = pd.read_csv(unified_csv)
df_combined = pd.concat([df_existing, df_metrics], ignore_index=True)
df_combined.to_csv(unified_csv, index=False)
print(f"Métricas unificadas actualizadas: {unified_csv} ({len(df_combined)} filas)")

assert df_metrics["auc_binary_depressive"].notna().all(), "AUC binario no debe ser NaN"
assert (df_metrics["split"].isin(["val", "test"])).all(), "split debe ser val o test"
print("OK — artefactos consistentes")

## 6. Visualizaciones: confusion matrix + training curves + leaderboard

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for idx, (split_name, df_split) in enumerate([("val", df_val), ("test", df_test)]):
    texts = df_split["text_clean"].astype(str).tolist()
    labels_mapped = [_label2id[l] for l in df_split["label"].astype(int).tolist()]
    ds = _TextDS(texts, labels_mapped, _tok, _max_len)
    out = _trainer.predict(ds)
    y_true = df_split["label"].astype(int).tolist()
    y_pred = [_id2label[int(p)] for p in np.argmax(out.predictions, axis=1)]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 2])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[idx],
                xticklabels=[0, 2], yticklabels=[0, 2])
    axes[idx].set_title(f"BETO multi — {split_name}")
    axes[idx].set_xlabel("Predicho")
    axes[idx].set_ylabel("Real")
plt.tight_layout()
fig_path = FIG_DIR / "beto_multi_confusion_matrix.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Figura guardada en {fig_path}")

In [ ]:
train_steps, train_loss = [], []
eval_epochs, eval_loss, eval_f1 = [], [], []
for entry in log_history:
    if "loss" in entry and "epoch" in entry and "eval_loss" not in entry:
        train_steps.append(entry.get("step", 0))
        train_loss.append(entry["loss"])
    if "eval_loss" in entry:
        eval_epochs.append(entry["epoch"])
        eval_loss.append(entry["eval_loss"])
        eval_f1.append(entry.get("eval_f1_macro", np.nan))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(train_steps, train_loss, color="#5bc0de", label="train loss")
axes[0].plot(eval_epochs, eval_loss, color="#d9534f", marker="o", label="val loss")
axes[0].set_xlabel("Step / Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("BETO multi — training & validation loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(eval_epochs, eval_f1, color="#5cb85c", marker="o", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("F1-macro (val)")
axes[1].set_title("BETO multi — validation F1-macro per epoch")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
fig_path = FIG_DIR / "beto_multi_training_curves.png"
plt.savefig(fig_path, dpi=300)
plt.show()
print(f"Figura guardada en {fig_path}")

In [ ]:
print("\n=== Leaderboard (test, comparando mono vs multi) ===")
df_all = pd.read_csv(TAB_DIR / "all_models_metrics.csv")
df_test_lb = df_all[df_all["split"] == "test"].sort_values("f1_macro", ascending=False)
print(df_test_lb[["model","feature_set","f1_macro","auc_binary_depressive","f1_class_2_depressive"]].to_string(index=False))

## 7. Conclusiones

- **BETO fine-tuned multi-corpus** sobre train balanceado (~500k-700k filas esperadas), 3 épocas, lr=2e-5, batch=16, fp16 en RTX 3060 Ti.
- **Tiempo total**: ~`{train_min:.1f}` min (registrado en el log).
- **Artefactos**: `models/beto_multi/best_model/{config.json, pytorch_model.bin, tokenizer.*}` + `model_meta.json` + `trainer_log_history.json`.
- **Métricas unificadas**: append 2 filas a `reports/tables/all_models_metrics.csv` (val + test) con `model="beto_multi"`.
- **Figuras**: `reports/figures/beto_multi_confusion_matrix.png` + `beto_multi_training_curves.png`.
- **Comparación**: BETO mono vs BETO multi — esperado: multi generaliza mejor cross-domain.